In [1]:
import pandas as pd
import vivarium_inputs
import gbd_mapping
import pathlib
from lsff_utils import config_utils
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_cause_and_scenario

Config: 'input_data:
    cache_data:
        base: True
    intermediary_data_cache_path:
        base: /share/scratch/users/{username}/cache'
Cache Dir: '/share/scratch/users/zmbc/cache'


In [2]:
location = "india"
vehicle = "rice"

In [3]:
# Parameters
location = "ethiopia"
vehicle = "salt"


In [4]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
    .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
    .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention_25_nrv', 'intervention_100_nrv', 'zero', 'baseline']

In [5]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylls.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylls = pd.read_parquet(path)
else:
    pregnancy_ylls = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/ylls.parquet"
        ).assign(value=0),
        scenarios,
    )
pregnancy_ylls

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
0,ylls,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,1,intervention_25_nrv,191,0,0
1,ylls,cause,other_causes,other_causes,10_to_14,invalid,1,intervention_25_nrv,191,0,0
2,ylls,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,2,intervention_25_nrv,191,0,0
3,ylls,cause,other_causes,other_causes,10_to_14,invalid,2,intervention_25_nrv,191,0,0
4,ylls,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,3,intervention_25_nrv,191,0,0
...,...,...,...,...,...,...,...,...,...,...,...
538195,ylls,cause,other_causes,other_causes,95_plus,severe,3,baseline,134,0,0
538196,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,4,baseline,134,0,0
538197,ylls,cause,other_causes,other_causes,95_plus,severe,4,baseline,134,0,0
538198,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,5,baseline,134,0,0


In [6]:
pregnancy_ylls.groupby("scenario").random_seed.nunique()

scenario
baseline                200
intervention_100_nrv    200
intervention_25_nrv     200
zero                    200
Name: random_seed, dtype: int64

In [7]:
assert (pregnancy_ylls[pregnancy_ylls.value > 0].entity == "maternal_disorders").all()

In [8]:
pregnancy_ylls_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylls).pipe(
    lambda df: df[df.index.get_level_values("entity") == "maternal_disorders"]
)
pregnancy_ylls_by_scenario

scenario              entity              wealth_quintile
baseline              maternal_disorders  1                  0.0
                                          2                  0.0
                                          3                  0.0
                                          4                  0.0
                                          5                  0.0
intervention_100_nrv  maternal_disorders  1                  0.0
                                          2                  0.0
                                          3                  0.0
                                          4                  0.0
                                          5                  0.0
intervention_25_nrv   maternal_disorders  1                  0.0
                                          2                  0.0
                                          3                  0.0
                                          4                  0.0
                                

In [9]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylds = pd.read_parquet(path)
else:
    pregnancy_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

pregnancy_ylds

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
0,ylds,cause,all_causes,all_causes,10_to_14,invalid,1,intervention_25_nrv,191,0,0
1,ylds,cause,pregnancy,pregnant,10_to_14,invalid,1,intervention_25_nrv,191,0,0
2,ylds,cause,pregnancy,parturition,10_to_14,invalid,1,intervention_25_nrv,191,0,0
3,ylds,cause,pregnancy,postpartum,10_to_14,invalid,1,intervention_25_nrv,191,0,0
4,ylds,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,1,intervention_25_nrv,191,0,0
...,...,...,...,...,...,...,...,...,...,...,...
1883695,ylds,cause,pregnancy,parturition,95_plus,severe,5,baseline,134,0,0
1883696,ylds,cause,pregnancy,postpartum,95_plus,severe,5,baseline,134,0,0
1883697,ylds,cause,maternal_disorders,maternal_disorders,95_plus,severe,5,baseline,134,0,0
1883698,ylds,cause,maternal_hemorrhage,maternal_hemorrhage,95_plus,severe,5,baseline,134,0,0


In [10]:
# Pregnancy has no disability, and maternal hemorrhage disability is counted in maternal_disorders
assert (
    pregnancy_ylds[
        pregnancy_ylds.entity.isin(["pregnancy", "maternal_hemorrhage"])
    ].value
    == 0
).all()

In [11]:
pregnancy_ylds_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylds).pipe(
    lambda df: df[
        ~df.index.get_level_values("entity").isin(["pregnancy", "maternal_hemorrhage"])
    ]
)
pregnancy_ylds_by_scenario

scenario              entity              wealth_quintile
baseline              anemia              1                  0.0
                                          2                  0.0
                                          3                  0.0
                                          4                  0.0
                                          5                  0.0
                      maternal_disorders  1                  0.0
                                          2                  0.0
                                          3                  0.0
                                          4                  0.0
                                          5                  0.0
intervention_100_nrv  anemia              1                  0.0
                                          2                  0.0
                                          3                  0.0
                                          4                  0.0
                                

In [12]:
pregnancy_dalys_by_scenario = pregnancy_ylls_by_scenario.add(
    pregnancy_ylds_by_scenario, fill_value=0
)
pregnancy_dalys_by_scenario

scenario              entity              wealth_quintile
baseline              anemia              1                  0.0
                                          2                  0.0
                                          3                  0.0
                                          4                  0.0
                                          5                  0.0
                      maternal_disorders  1                  0.0
                                          2                  0.0
                                          3                  0.0
                                          4                  0.0
                                          5                  0.0
intervention_100_nrv  anemia              1                  0.0
                                          2                  0.0
                                          3                  0.0
                                          4                  0.0
                                

In [13]:
ylds_path = f"results/rescaled_child_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(ylds_path).is_file():
    assert len(pd.read_parquet(ylds_path)) == 0

In [14]:
path = f"results/rescaled_child_results/{vehicle}/{location}/ylls.parquet"
if pathlib.Path(path).is_file():
    neonatal_ylls = pd.read_parquet(path).rename(
        columns={"maternal_scenario": "scenario"}
    )
else:
    neonatal_ylls = expand_to_all_scenarios(
        pd.read_parquet(f"results/rescaled_child_results/rice/india/ylls.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_ylls

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,random_seed,input_draw,value
0,ylls,cause,stillborn,stillborn,0_to_6_months,Female,1,baseline,intervention_25_nrv,20,0,0
1,ylls,cause,stillborn,stillborn,0_to_6_months,Female,2,baseline,intervention_25_nrv,20,0,0
2,ylls,cause,stillborn,stillborn,0_to_6_months,Female,3,baseline,intervention_25_nrv,20,0,0
3,ylls,cause,stillborn,stillborn,0_to_6_months,Female,4,baseline,intervention_25_nrv,20,0,0
4,ylls,cause,stillborn,stillborn,0_to_6_months,Female,5,baseline,intervention_25_nrv,20,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
47835,ylls,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,baseline,139,0,0
47836,ylls,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,baseline,139,0,0
47837,ylls,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,baseline,139,0,0
47838,ylls,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,baseline,139,0,0


In [15]:
neonatal_ylls_by_scenario = aggregate_by_cause_and_scenario(neonatal_ylls)
assert (
    neonatal_ylls_by_scenario[
        neonatal_ylls_by_scenario.index.get_level_values("entity") != "other_causes"
    ]
    == 0
).all()
neonatal_ylls_by_scenario = neonatal_ylls_by_scenario[
    neonatal_ylls_by_scenario.index.get_level_values("entity") == "other_causes"
]
neonatal_ylls_by_scenario = (
    neonatal_ylls_by_scenario.reset_index()
    .assign(entity="lbwsg")
    .set_index(neonatal_ylls_by_scenario.index.names)
    .value
)
neonatal_ylls_by_scenario

scenario              entity  wealth_quintile
baseline              lbwsg   1                  0.0
                              2                  0.0
                              3                  0.0
                              4                  0.0
                              5                  0.0
intervention_100_nrv  lbwsg   1                  0.0
                              2                  0.0
                              3                  0.0
                              4                  0.0
                              5                  0.0
intervention_25_nrv   lbwsg   1                  0.0
                              2                  0.0
                              3                  0.0
                              4                  0.0
                              5                  0.0
zero                  lbwsg   1                  0.0
                              2                  0.0
                              3                  0.0


In [16]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_ylds = pd.read_parquet(path)
else:
    non_pregnancy_anemia_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"../0400_non_pregnant_anemia_model/results/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_ylds

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,273.390021,zero
1,Female,0.0,0.019178,2,191.851401,zero
2,Female,0.0,0.019178,3,139.900681,zero
3,Female,0.0,0.019178,4,147.088093,zero
4,Female,0.0,0.019178,5,79.362233,zero
...,...,...,...,...,...,...
995,Male,95.0,125.000000,3,26.890065,intervention_25_nrv
996,Male,95.0,125.000000,4,27.038454,intervention_100_nrv
997,Male,95.0,125.000000,4,27.038454,intervention_25_nrv
998,Male,95.0,125.000000,5,21.509160,intervention_100_nrv


In [17]:
# For comparison with previous round of results, we also look at
# WRA and U5
wra_non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[
        (non_pregnancy_anemia_ylds.sex == "Female")
        & (non_pregnancy_anemia_ylds.age_start >= 10)
        & (non_pregnancy_anemia_ylds.age_end <= 55)
    ].assign(entity="anemia", input_draw="draw_0")
)
wra_non_pregnancy_anemia_ylds_by_scenario

scenario              entity  wealth_quintile
baseline              anemia  1                  68691.469785
                              2                  42172.723761
                              3                  37584.603080
                              4                  33877.907876
                              5                  29845.438250
intervention_100_nrv  anemia  1                  61812.093710
                              2                  36365.537538
                              3                  31267.140746
                              4                  31333.033823
                              5                  27884.210512
intervention_25_nrv   anemia  1                  66850.875064
                              2                  40570.367655
                              3                  35802.943787
                              4                  33208.919643
                              5                  29335.039359
zero                  an

In [18]:
scenarios[1]

'intervention_100_nrv'

In [19]:
(
    wra_non_pregnancy_anemia_ylds_by_scenario.loc["baseline"].sum()
    + pregnancy_ylds_by_scenario.loc[("baseline", "anemia")].sum()
) - (
    wra_non_pregnancy_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
    + pregnancy_ylds_by_scenario.loc[(scenarios[1], "anemia")].sum()
)

23510.12642313371

In [20]:
u5_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[(non_pregnancy_anemia_ylds.age_end <= 5)].assign(
        entity="anemia", input_draw="draw_0"
    )
)
u5_anemia_ylds_by_scenario

scenario              entity  wealth_quintile
baseline              anemia  1                  106520.238000
                              2                   68839.521005
                              3                   49355.671545
                              4                   49163.393194
                              5                   29171.375430
intervention_100_nrv  anemia  1                  106520.238000
                              2                   68839.521005
                              3                   49355.671545
                              4                   49163.393194
                              5                   29171.375430
intervention_25_nrv   anemia  1                  106520.238000
                              2                   68839.521005
                              3                   49355.671545
                              4                   49163.393194
                              5                   29171.375430
zero     

In [21]:
(
    u5_anemia_ylds_by_scenario.loc["baseline"].sum()
    - u5_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
)

0.0

In [22]:
non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_anemia_ylds_by_scenario

scenario              entity  wealth_quintile
baseline              anemia  1                  289900.901395
                              2                  172510.203320
                              3                  135456.055436
                              4                  131313.885430
                              5                   85349.300638
intervention_100_nrv  anemia  1                  280321.340656
                              2                  164954.076873
                              3                  127370.513364
                              4                  128073.579511
                              5                   82969.402239
intervention_25_nrv   anemia  1                  287323.401779
                              2                  170417.250617
                              3                  133168.741226
                              4                  130461.215681
                              5                   84729.904041
zero     

In [23]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ylls_by_scenario.csv"
if pathlib.Path(path).is_file():
    neural_tube_defect_ylls_by_scenario = pd.read_csv(path)
else:
    neural_tube_defect_ylls_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/intervention/ylls_by_scenario.csv"
        ).assign(value=0),
        scenarios,
    )

neural_tube_defect_ylls_by_scenario = neural_tube_defect_ylls_by_scenario.set_index(
    ["scenario", "entity", "wealth_quintile"]
).value
neural_tube_defect_ylls_by_scenario

scenario              entity  wealth_quintile
zero                  ntd     1                  155638.819702
                              2                  148048.117342
                              3                  134749.677549
                              4                  118744.475674
                              5                   93677.505091
baseline              ntd     1                  155638.819702
                              2                  148048.117342
                              3                  134749.677549
                              4                  118744.475674
                              5                   93677.505091
intervention_25_nrv   ntd     1                   83496.309791
                              2                   75163.085269
                              3                   61826.673345
                              4                   85542.431965
                              5                   74597.973512
intervent

In [24]:
dalys_by_scenario = (
    pregnancy_dalys_by_scenario.add(neonatal_ylls_by_scenario, fill_value=0)
    .add(non_pregnancy_anemia_ylds_by_scenario, fill_value=0)
    .add(neural_tube_defect_ylls_by_scenario, fill_value=0)
)
dalys_by_scenario

scenario  entity  wealth_quintile
baseline  anemia  1                  289900.901395
                  2                  172510.203320
                  3                  135456.055436
                  4                  131313.885430
                  5                   85349.300638
                                         ...      
zero      ntd     1                  155638.819702
                  2                  148048.117342
                  3                  134749.677549
                  4                  118744.475674
                  5                   93677.505091
Name: value, Length: 80, dtype: float64

In [25]:
import pathlib

In [26]:
path = f"./results/{location}/{vehicle}/dalys_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
dalys_by_scenario.to_csv(path)